# 因子构建 · 第一步：读取数据与构建目标变量 y

1. 读取 **Data** 目录下所有数据文件
2. **规整数据类型**：日期列统一、数值列统一
3. 用 **上证50 与 上证1000 的涨跌幅做差** 作为测因子的 **y**（不做 z-score）

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# 导入项目根目录的 config（项目根 = 含 config.py 的目录）
_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_build' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# 路径与列名：从 config 读取（可改 config.py 统一调整）
BASE_DIR = config.get_base_dir()
DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
PCT_COL = config.PCT_COL
print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)
print('Data 存在:', os.path.isdir(DATA_DIR))

BASE_DIR: /Users/amadeus/Desktop/p3_adjusted_program
DATA_DIR: /Users/amadeus/Desktop/p3_adjusted_program/Data
Data 存在: True


## 1. 扫描并读取 Data 下所有 Excel 文件

In [2]:
# 列出 Data 下 Excel 文件（排除临时文件，见 config.SKIP_FILE_PREFIX）
all_files = [f for f in os.listdir(DATA_DIR) 
             if any(f.endswith(ext) for ext in config.EXCEL_EXTENSIONS) 
             and not f.startswith(config.SKIP_FILE_PREFIX)]
all_files.sort()
print('Data 下 Excel 文件:', all_files)

Data 下 Excel 文件: ['因子_国家队_pair_zscore.xlsx', '因子_国家队_增强版_pair_zscore.xlsx', '数据_波动率与股债指数.xlsx', '行情_上证1000.xlsx', '行情_上证50.xlsx', '行情_沪深300.xlsx']


In [3]:
def normalize_dtypes(df: pd.DataFrame, date_columns=None):
    """规整数据类型：日期列转 date，数值列转 float。"""
    df = df.copy()
    # 常见日期列名
    date_candidates = date_columns or [config.DATE_COL, 'Date', '日期', 'date']
    for col in df.columns:
        if col in date_candidates:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.date
            continue
        # 数值列：先去掉千分位逗号再转 float（Excel 常存为 "3,073.67"）
        if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
            s = df[col].astype(str).str.replace(',', '').str.strip()
            df[col] = pd.to_numeric(s, errors='coerce')
    return df


# 读取所有文件，按「单 sheet」或「多 sheet」分别存储
raw_data = {}   # 文件名 -> 单 DataFrame 或 dict[sheet_name -> DataFrame]

for f in all_files:
    path = os.path.join(DATA_DIR, f)
    try:
        xl = pd.ExcelFile(path)
        if len(xl.sheet_names) == 1:
            df = pd.read_excel(path, sheet_name=0)
            df = normalize_dtypes(df)
            raw_data[f] = df
        else:
            sheets = pd.read_excel(path, sheet_name=None)
            raw_data[f] = {
                name: normalize_dtypes(sh) 
                for name, sh in sheets.items()
            }
    except Exception as e:
        print(f'读取失败 {f}:', e)

print('已读取文件数:', len(raw_data))
for k, v in raw_data.items():
    if isinstance(v, dict):
        print(f'  {k}: 多 sheet -> {list(v.keys())}')
    else:
        print(f'  {k}: 单表, shape={v.shape}, columns={list(v.columns)[:8]}...' if len(v.columns) > 8 else f'  {k}: 单表, shape={v.shape}, columns={list(v.columns)}')

已读取文件数: 6
  因子_国家队_pair_zscore.xlsx: 多 sheet -> ['pct_chg', 'netinflow', 'NAV_iopv_discount', 'iopv', 'turn', 'amt_btin', 'amt', 'volume', 'volume_btin', 'ETF申购赎回现金差额', 'Target']
  因子_国家队_增强版_pair_zscore.xlsx: 多 sheet -> ['pct_chg', 'unit_total', 'netinflow', 'NAV_iopv_discount', 'iopv', 'turn', 'amt_btin', 'amt', 'volume', 'volume_btin', 'ETF申购赎回现金差额', 'Target']
  数据_波动率与股债指数.xlsx: 多 sheet -> [' 国债收益率', '汇率收盘价', '中国波动率收盘价', '中证股债风险平均指数收盘价', '大宗商品收盘价']
  行情_上证1000.xlsx: 单表, shape=(2430, 12), columns=['交易日期', '开盘点位', '最高点位', '最低点位', '收盘价', '涨跌', '涨跌幅(%)', '开始日累计涨跌']...
  行情_上证50.xlsx: 单表, shape=(2431, 12), columns=['交易日期', '开盘点位', '最高点位', '最低点位', '收盘价', '涨跌', '涨跌幅(%)', '开始日累计涨跌']...
  行情_沪深300.xlsx: 单表, shape=(2431, 12), columns=['交易日期', '开盘点位', '最高点位', '最低点位', '收盘价', '涨跌', '涨跌幅(%)', '开始日累计涨跌']...


## 2. 规整后数据概览（列名与类型）

In [4]:
# 从 raw_data 的键中匹配行情文件（关键词见 config.FILE_KEYWORDS_CSI50 / CSI1000）
def _single_sheet(k):
    return isinstance(raw_data.get(k), pd.DataFrame)
def _match(k, keywords):
    return all(kw in k for kw in keywords)
filename_50 = next((k for k in raw_data if _single_sheet(k) and _match(k, config.FILE_KEYWORDS_CSI50)), None)
filename_1000 = next((k for k in raw_data if _single_sheet(k) and _match(k, config.FILE_KEYWORDS_CSI1000)), None)
filename_300 = next((k for k in raw_data if _single_sheet(k) and '沪深300' in k and '行情' in k), None)
file_csi50 = [filename_50] if filename_50 else []
file_csi1000 = [filename_1000] if filename_1000 else []
file_csi300 = [filename_300] if filename_300 else []

print('上证50 行情文件:', file_csi50)
print('上证1000 行情文件:', file_csi1000)
print('沪深300 行情文件:', file_csi300)

# 展示行情表列名，确认是否有「涨跌幅」
if file_csi50:
    d = raw_data[file_csi50[0]]
    print('\n上证50 表列名:', list(d.columns))
if file_csi1000:
    d = raw_data[file_csi1000[0]]
    print('上证1000 表列名:', list(d.columns))

上证50 行情文件: ['行情_上证50.xlsx']
上证1000 行情文件: ['行情_上证1000.xlsx']
沪深300 行情文件: ['行情_沪深300.xlsx']

上证50 表列名: ['交易日期', '开盘点位', '最高点位', '最低点位', '收盘价', '涨跌', '涨跌幅(%)', '开始日累计涨跌', '开始日累计涨跌幅', '成交量(万股)', '成交额(万元)', '持仓量']
上证1000 表列名: ['交易日期', '开盘点位', '最高点位', '最低点位', '收盘价', '涨跌', '涨跌幅(%)', '开始日累计涨跌', '开始日累计涨跌幅', '成交量(万股)', '成交额(万元)', '持仓量']


## 3. 构建 y：上证50涨跌幅 − 上证1000涨跌幅

In [5]:
# 列名已从 config 读取（DATE_COL, PCT_COL）

def get_pct_series(raw_data, filename, date_col=DATE_COL, pct_col=PCT_COL):
    """从 raw_data 中取指定文件的 交易日期、涨跌幅，并规整类型。"""
    if filename not in raw_data:
        print(f'未在 raw_data 中找到: {filename}，当前键: {list(raw_data.keys())}')
        return None
    obj = raw_data[filename]
    if isinstance(obj, dict):
        print(f'{filename} 为多 sheet，无法取单表')
        return None
    df = obj.copy()
    df.columns = df.columns.str.strip() if hasattr(df.columns, 'str') else df.columns
    if date_col not in df.columns and 'Date' in df.columns:
        df = df.rename(columns={'Date': date_col})
    if date_col not in df.columns:
        print(f'未找到日期列，可选: {list(df.columns)}')
        return None
    if pct_col not in df.columns:
        for c in [PCT_COL, '涨跌幅', 'pct_chg']:
            if c in df.columns:
                df = df.rename(columns={c: pct_col})
                break
        if pct_col not in df.columns:
            print(f'未找到涨跌幅列，可选: {list(df.columns)}')
            return None
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce').dt.date
    s = df[pct_col]
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype(str).str.replace(',', '').str.strip()
    df[pct_col] = pd.to_numeric(s, errors='coerce')
    return df[[date_col, pct_col]].dropna(subset=[date_col, pct_col])

if not file_csi50 or not file_csi1000:
    raise FileNotFoundError('缺少上证50或上证1000行情文件，请检查 Data 目录')

df_50 = get_pct_series(raw_data, file_csi50[0])
df_1000 = get_pct_series(raw_data, file_csi1000[0])
if df_50 is None or df_1000 is None or len(df_50) == 0 or len(df_1000) == 0:
    print('诊断: df_50 行数=', 0 if df_50 is None else len(df_50), ', df_1000 行数=', 0 if df_1000 is None else len(df_1000))
    raise ValueError('取数后行为 0，请检查 Data 下行情文件')
print('取数后: 上证50 行数=', len(df_50), ', 上证1000 行数=', len(df_1000))
df_50 = df_50.rename(columns={PCT_COL: '涨跌幅_50'})
df_1000 = df_1000.rename(columns={PCT_COL: '涨跌幅_1000'})

df_50['_date'] = pd.to_datetime(df_50[DATE_COL]).dt.strftime('%Y-%m-%d')
df_1000['_date'] = pd.to_datetime(df_1000[DATE_COL]).dt.strftime('%Y-%m-%d')
df_price = df_50[['_date','涨跌幅_50']].merge(
    df_1000[['_date','涨跌幅_1000']], on='_date', how='inner')
df_price[DATE_COL] = pd.to_datetime(df_price['_date']).dt.date
df_price = df_price.drop(columns=['_date']).sort_values(DATE_COL).reset_index(drop=True)

# 样本截止：若 config.CUTOFF_DATE 有设置则只保留该日期及之前
if getattr(config, 'CUTOFF_DATE', None) is not None:
    import datetime
    cut = config.CUTOFF_DATE
    if isinstance(cut, (tuple, list)) and len(cut) >= 3:
        cut = datetime.date(int(cut[0]), int(cut[1]), int(cut[2]))
    df_price = df_price[df_price[DATE_COL] <= cut].reset_index(drop=True)
    print('样本截止:', cut)

# 相减前：空值 drop
before_drop = len(df_price)
df_price = df_price.dropna(subset=['涨跌幅_50', '涨跌幅_1000'])
df_price = df_price.reset_index(drop=True)

# y = 上证50涨跌幅 − 上证1000涨跌幅（不做 z-score）
df_price['y'] = df_price['涨跌幅_50'] - df_price['涨跌幅_1000']

print('y = 上证50涨跌幅 - 上证1000涨跌幅（大盘减小盘）')
print('已剔除空值交易日（剔除前 {} 行，剔除后 {} 行）'.format(before_drop, len(df_price)))
print('【50-1000 因子】有效值:', len(df_price))
print('日期范围:', df_price[DATE_COL].min(), '~', df_price[DATE_COL].max())
print('y 统计: mean={:.4f}, std={:.4f}'.format(df_price['y'].mean(), df_price['y'].std()))
display(df_price.head(10))

取数后: 上证50 行数= 2431 , 上证1000 行数= 2430
y = 上证50涨跌幅 - 上证1000涨跌幅（大盘减小盘）
已剔除空值交易日（剔除前 2427 行，剔除后 2427 行）
【50-1000 因子】有效值: 2427
日期范围: 2015-11-19 ~ 2025-11-13
y 统计: mean=0.0149, std=1.2731


,涨跌幅_50,涨跌幅_1000,交易日期,y
0,1.19,3.09,2015-11-19,-1.90
1,-0.07,2.00,2015-11-20,-2.07
2,-0.40,-0.95,2015-11-23,0.55
3,-0.36,1.14,2015-11-24,-1.50
4,0.48,2.26,2015-11-25,-1.78
5,-0.35,-0.71,2015-11-26,0.36
6,-4.75,-6.47,2015-11-27,1.72
7,0.08,0.68,2015-11-30,-0.60
8,0.37,-0.46,2015-12-01,0.83
9,4.84,-1.34,2015-12-02,6.18


## 4. 汇总：规整后的数据与 y 表

In [6]:
# 将「规整后的原始数据」和「目标 y 表」保存到变量，供后续 notebook 使用
# raw_data: 所有 Data 下文件的规整结果
# df_y: 含 交易日期, 涨跌幅_50, 涨跌幅_1000, y（y = 涨跌幅_50 - 涨跌幅_1000）

df_y = df_price[['交易日期', '涨跌幅_50', '涨跌幅_1000', 'y']].copy()
print('df_y 已就绪，列:', list(df_y.columns))

# 输出时间序列到 factor_build/outputs（每次运行覆盖更新）
_out_dir = os.path.join(os.getcwd(), getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
os.makedirs(_out_dir, exist_ok=True)
_path_y = os.path.join(_out_dir, '01_y_timeseries.xlsx')
df_y.to_excel(_path_y, index=False)
print('已输出:', _path_y)
print('\n前 5 行:')
display(df_y.head())

df_y 已就绪，列: ['交易日期', '涨跌幅_50', '涨跌幅_1000', 'y']
已输出: /Users/amadeus/Desktop/p3_adjusted_program/factor_build/outputs/01_y_timeseries.xlsx

前 5 行:


,交易日期,涨跌幅_50,涨跌幅_1000,y
0,2015-11-19,1.19,3.09,-1.90
1,2015-11-20,-0.07,2.00,-2.07
2,2015-11-23,-0.40,-0.95,0.55
3,2015-11-24,-0.36,1.14,-1.50
4,2015-11-25,0.48,2.26,-1.78
